In [4]:
# ==============================================================================
# SEZIONE 1: SETUP ENVIROMENT
# ==============================================================================

import os
import sys
import numpy as np
import torch
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NoisyStudent_train' else NOTEBOOK_DIR

print(f"PyTorch: {torch.__version__}")
print(f"Project Root: {PROJECT_ROOT}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} {'(' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else ''}")

PyTorch: 2.10.0+cu128
Project Root: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25
Device: cuda (NVIDIA GeForce RTX 5060 Ti)


In [5]:
# ==============================================================================
# SEZIONE 2: LOAD TEACHER MODEL (RAVDESS best_swa_model)
# ==============================================================================

# Aggiungi project root al sys path per gli import
sys.path.insert(0, str(PROJECT_ROOT))

import config as Config
from models import get_model

# ---- PATH CONFIGURAZIONE ----
TEACHER_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "ravdess" / "best_swa_model.pth"

# Verifica che il checkpoint esiste
if not TEACHER_CHECKPOINT.exists():
    raise FileNotFoundError(f"❌ Teacher checkpoint non trovato: {TEACHER_CHECKPOINT}")

print(f"✅ Teacher checkpoint trovato: {TEACHER_CHECKPOINT}")

# ---- LOAD TEACHER MODEL ----
# Inizializza il modello con stessi iperparametri usati per RAVDESS training
teacher_model = get_model(
    model_name='CRNN_BiLSTM',  # Stesso modello usato per RAVDESS training
    batch_size=Config.BATCH_SIZE_RAVDESS,
    time_steps=Config.TIME_STEPS_RAVDESS,
    dropout=Config.DROPOUT_RAVDESS
).to(device)

# Carica i pesi del checkpoint SWA
teacher_model.load_state_dict(torch.load(TEACHER_CHECKPOINT, map_location=device))

# ---- CONFIGURAZIONE TEACHER PER INFERENCE ----
teacher_model.eval()  # Mode inference (no dropout, no batch norm updates)

# Disabilita gradient computation (non serve aggiornare i pesi)
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"\n📚 TEACHER MODEL LOADED")
print(f"   - Architecture: CRNN_BiLSTM")
print(f"   - Checkpoint: best_swa_model.pth (SWA averaged)")
print(f"   - Mode: Inference (eval mode)")
print(f"   - Requires Grad: False")
print(f"   - Device: {device}")
print(f"   - Total Parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
print("="*80 + "\n")

✅ Teacher checkpoint trovato: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\checkpoints\ravdess\best_swa_model.pth

📚 TEACHER MODEL LOADED
   - Architecture: CRNN_BiLSTM
   - Checkpoint: best_swa_model.pth (SWA averaged)
   - Mode: Inference (eval mode)
   - Requires Grad: False
   - Device: cuda
   - Total Parameters: 1,137,029



In [6]:
# ==============================================================================
# SEZIONE 3: GENERATE SOFT LABELS (Teacher inference on IEMOCAP)
# ==============================================================================
# Genera soft labels (probabilità del teacher) per il dataset IEMOCAP (solo train e validation splits)
# Questi soft labels verranno usati per il training dello student con 
# Knowledge Distillation (KL divergence loss)
# ==============================================================================

from dataset.custom_iemocap_dataset import CustomIEMOCAPDataset
from torch.utils.data import DataLoader
import torch.nn.functional as F

# ---- LOAD IEMOCAP DATASET ----
print("📂 Loading IEMOCAP dataset...")
iemocap_path = PROJECT_ROOT / Config.IEMOCAP_PATH

if not iemocap_path.exists():
    raise FileNotFoundError(f"❌ IEMOCAP dataset not found: {iemocap_path}")

# Creamo dataset per train, validation, test splits
train_iemocap_dataset = CustomIEMOCAPDataset(
    dataset_root=str(iemocap_path),
    split='train',
    spec_freq_mask=Config.SPEC_FREQ_MASK_IEMOCAP,
    spec_time_mask=Config.SPEC_TIME_MASK_IEMOCAP
)

val_iemocap_dataset = CustomIEMOCAPDataset(
    dataset_root=str(iemocap_path),
    split='validation'
)

test_iemocap_dataset = CustomIEMOCAPDataset(
    dataset_root=str(iemocap_path),
    split='test'
)

print(f"   ✅ Train samples: {len(train_iemocap_dataset)}")
print(f"   ✅ Val samples: {len(val_iemocap_dataset)}")
print(f"   ✅ Test samples: {len(test_iemocap_dataset)}")

# ---- CREATE DATALOADERS ----
train_iemocap_loader = DataLoader(
    train_iemocap_dataset,
    batch_size=Config.BATCH_SIZE_IEMOCAP,
    shuffle=False  # Important: no shuffle per mantenere l'ordine
)

val_iemocap_loader = DataLoader(
    val_iemocap_dataset,
    batch_size=Config.BATCH_SIZE_IEMOCAP,
    shuffle=False
)

test_iemocap_loader = DataLoader(
    test_iemocap_dataset,
    batch_size=Config.BATCH_SIZE_IEMOCAP,
    shuffle=False
)

print(f"\n✅ DataLoaders created")
print(f"   - Batch size: {Config.BATCH_SIZE_IEMOCAP}")
print(f"   - Shuffle: False (per mantenere coerenza soft labels)")

# ---- FUNZIONE PER GENERARE SOFT LABELS ----
def generate_soft_labels(teacher, dataloader, device, temperature=Config.TEMPERATURE):
    """
    Genera soft labels (soft targets) usando il teacher model.
    
    Args:
        teacher: Teacher model in eval mode
        dataloader: DataLoader per le features
        device: torch device
        temperature: Temperature per il soft target scaling
    
    Returns:
        soft_labels_dict: Dict con {idx: soft_probs} per ogni sample
        hard_labels_list: Lista delle hard labels (ground truth)
        logits_list: Lista dei logit raw dal teacher
    """
    teacher.eval()
    soft_labels_dict = {}
    hard_labels_list = []
    logits_list = []
    all_probs = []
    
    sample_idx = 0
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            audio_features = batch['audio_features'].to(device)
            emotion_ids = batch['emotion_id'].to(device)
            
            # Forward pass attraverso il teacher
            logits = teacher(audio_features)  # shape: (batch_size, num_classes)
            
            # Calcola soft probabilities con temperature scaling
            # Temperature BASSA → più sharp (concentrato su argmax)
            # Temperature ALTA → più smooth (distribuito)
            soft_probs = F.softmax(logits / temperature, dim=1)  # Scaled by T
            
            # Salva logit e probabilità
            logits_list.extend(logits.cpu().numpy())
            all_probs.extend(soft_probs.cpu().numpy())
            
            # Salva hard labels
            hard_labels_list.extend(emotion_ids.cpu().numpy())
            
            # Crea mapping: sample_idx → soft_probs
            for i in range(audio_features.shape[0]):
                soft_labels_dict[sample_idx] = soft_probs[i].cpu().numpy()
                sample_idx += 1
            
            # Progress bar
            if (batch_idx + 1) % 10 == 0:
                print(f"   Processed {sample_idx} samples...")
    
    return soft_labels_dict, hard_labels_list, logits_list, all_probs

# ---- GENERA SOFT LABELS PER TUTTI I SPLIT ----
print("\n" + "="*80)
print("🔄 GENERATING SOFT LABELS")
print("="*80)

print("\n📊 Train split:")
train_soft_labels, train_hard_labels, train_logits, train_all_probs = generate_soft_labels(
    teacher_model, train_iemocap_loader, device
)

print("\n📊 Validation split:")
val_soft_labels, val_hard_labels, val_logits, val_all_probs = generate_soft_labels(
    teacher_model, val_iemocap_loader, device
)


# ---- STATISTICHE SOFT LABELS ----
print("\n" + "="*80)
print("📈 SOFT LABELS STATISTICS")
print("="*80)

import numpy as np

# Analizza confidenza del teacher (max probability per sample)
train_confidences = np.max(train_all_probs, axis=1)
val_confidences = np.max(val_all_probs, axis=1)


print(f"\n🎯 TEACHER CONFIDENCE (max softmax probability):")
print(f"   Train - Mean: {train_confidences.mean():.4f}, Std: {train_confidences.std():.4f}")
print(f"           Min: {train_confidences.min():.4f}, Max: {train_confidences.max():.4f}")
print(f"   Val   - Mean: {val_confidences.mean():.4f}, Std: {val_confidences.std():.4f}")
print(f"           Min: {val_confidences.min():.4f}, Max: {val_confidences.max():.4f}")


# ---- VERIFICA: CONFRONTO HARD vs SOFT LABELS ----
print(f"\n✅ HARD vs SOFT LABELS ALIGNMENT:")

emotion_names = ['Neutral', 'Happy', 'Sad', 'Angry']

for split_name, hard_labels, soft_probs in [
    ('Train', train_hard_labels, train_all_probs),
    ('Val', val_hard_labels, val_all_probs)
]:
    # Calcola accuratezza: hard label == argmax(soft_label)
    soft_preds = np.argmax(soft_probs, axis=1)
    hard_accuracy = np.mean(soft_preds == hard_labels)
    
    print(f"\n   {split_name} split:")
    print(f"     - Soft labels agree with hard: {hard_accuracy*100:.2f}%")
    print(f"     - Numero samples: {len(hard_labels)}")

# ---- ESTRAI HARD LABELS PER TEST SET (NO soft labels) ----
print(f"\n📊 Test split (NO soft labels - PURE):")
test_hard_labels = []
with torch.no_grad():
    for batch in test_iemocap_loader:
        test_hard_labels.extend(batch['emotion_id'].cpu().numpy())

print(f"   - test_hard_labels: {len(test_hard_labels)} entries (NO soft labels)")

print("\n" + "="*80)
print("✅ SOFT LABELS GENERATED AND STORED")
print(f"   - train_soft_labels: {len(train_soft_labels)} entries")
print(f"   - val_soft_labels: {len(val_soft_labels)} entries")
print(f"   - test_hard_labels: {len(test_hard_labels)} entries (NO soft labels)")
print("="*80 + "\n")

📂 Loading IEMOCAP dataset...
✅ Caricate 5531 etichette
🔍 Raccogliendo campioni audio...
✅ Raccolti 2943 campioni audio validi
   - Solo campioni improvvisati
   - Emozioni: ['neutral', 'happy', 'sad', 'angry', 'happy']
📊 Statistiche del dataset IEMOCAP:

📊 ANALISI IEMOCAP TRAINING SET

🔹 SAMPLES TOTALI: 1678
🔹 SESSIONI: ['1', '2', '3']
🔹 SPEAKER UNICI (session, gender): 6
   Elenco: [('1', 'F'), ('1', 'M'), ('2', 'F'), ('2', 'M'), ('3', 'F'), ('3', 'M')]
🔹 IMPROVVISAZIONI UNICHE: 12
   Elenco: ['01', '02', '03', '04', '05', '05a', '05b', '06', '07', '08', '08a', '08b']

👥 SPEAKER INDEPENDENCE (per verificare leakage):
   - Sessione 1: (Ses1, F), (Ses1, M)
   - Sessione 2: (Ses2, F), (Ses2, M)
   - Sessione 3: (Ses3, F), (Ses3, M)

🎭 DISTRIBUZIONE EMOZIONI:
   - Angry     :  174 ( 10.4%) ██
   - Happy     :  472 ( 28.1%) █████
   - Neutral   :  638 ( 38.0%) ███████
   - Sad       :  394 ( 23.5%) ████

📋 DISTRIBUZIONE CAMPIONI PER SESSIONE:
   - Sessione 1:  521 ( 31.0%) ██████
      └─ 

In [9]:
# ==============================================================================
# SEZIONE 4: CUSTOM DATASET WRAPPER CON SOFT LABELS
# ==============================================================================
# Crea dataset wrapper che ritorna:
#   - TRAIN/VAL: (audio_features, hard_labels, soft_labels) per KD training
#   - TEST: (audio_features, hard_labels) PURO - senza soft labels
#
# Questa separazione è FONDAMENTALE:
# - Train/Val usano soft labels dal teacher per il Knowledge Distillation
# - Test rimane incontaminato per valutare la REALE performance dello student
# ==============================================================================

from torch.utils.data import Dataset

# ---- DATASET CON SOFT LABELS (per Train e Validation) ----
class NoisyStudentDatasetWithSoftLabels(Dataset):
    """
    Custom Dataset che aggiunge soft labels dal teacher ai dati IEMOCAP.
    
    Ritorna una tripla per ogni sample:
        (audio_features, hard_label, soft_label)
    
    Usato per: TRAIN e VALIDATION splits
    """
    
    def __init__(self, iemocap_dataset, soft_labels_dict, hard_labels_list):
        """
        Args:
            iemocap_dataset: CustomIEMOCAPDataset object
            soft_labels_dict: Dict con {sample_idx: soft_probs_array}
            hard_labels_list: Lista con hard labels (ground truth)
        """
        self.iemocap_dataset = iemocap_dataset
        self.soft_labels_dict = soft_labels_dict
        self.hard_labels_list = hard_labels_list
        
        # Verifica coerenza
        assert len(self) == len(hard_labels_list), \
            f"Dataset size mismatch: {len(self)} vs {len(hard_labels_list)}"
        assert len(self.soft_labels_dict) == len(hard_labels_list), \
            f"Soft labels mismatch: {len(self.soft_labels_dict)} vs {len(hard_labels_list)}"
    
    def __len__(self):
        return len(self.iemocap_dataset)
    
    def __getitem__(self, idx):
        # Ottieni dati originali dal dataset IEMOCAP
        batch = self.iemocap_dataset[idx]
        
        audio_features = batch['audio_features']
        hard_label = torch.tensor(self.hard_labels_list[idx], dtype=torch.long)
        soft_label = torch.tensor(self.soft_labels_dict[idx], dtype=torch.float32)
        
        return {
            'audio_features': audio_features,
            'hard_label': hard_label,
            'soft_label': soft_label,
            'emotion_id': hard_label  # Per compatibilità con eval script
        }


# ---- DATASET SENZA SOFT LABELS (per Test - PURO) ----
class NoisyStudentDatasetNoSoftLabels(Dataset):
    """
    Custom Dataset PURO senza soft labels dal teacher.
    Usato SOLO per il TEST SET.
    
    Ritorna:
        (audio_features, hard_label)
    
    Questo rimane incontaminato per valutare la REALE performance dello student.
    """
    
    def __init__(self, iemocap_dataset, hard_labels_list):
        """
        Args:
            iemocap_dataset: CustomIEMOCAPDataset object
            hard_labels_list: Lista con hard labels (ground truth)
        """
        self.iemocap_dataset = iemocap_dataset
        self.hard_labels_list = hard_labels_list
        
        # Verifica coerenza
        assert len(self) == len(hard_labels_list), \
            f"Dataset size mismatch: {len(self)} vs {len(hard_labels_list)}"
    
    def __len__(self):
        return len(self.iemocap_dataset)
    
    def __getitem__(self, idx):
        # Ottieni dati originali dal dataset IEMOCAP
        batch = self.iemocap_dataset[idx]
        
        audio_features = batch['audio_features']
        hard_label = torch.tensor(self.hard_labels_list[idx], dtype=torch.long)
        
        return {
            'audio_features': audio_features,
            'hard_label': hard_label,
            'emotion_id': hard_label  # Per compatibilità con eval script
        }


# ---- CREA I DATASET WRAPPER ----
print("\n" + "="*80)
print("🔧 CREATING CUSTOM DATASET WRAPPERS FOR NOISY STUDENT")
print("="*80)

# Train dataset: CON soft labels
train_dataset_kd = NoisyStudentDatasetWithSoftLabels(
    iemocap_dataset=train_iemocap_dataset,
    soft_labels_dict=train_soft_labels,
    hard_labels_list=train_hard_labels
)

print(f"\n✅ Train dataset (with soft labels):")
print(f"   - Size: {len(train_dataset_kd)}")
print(f"   - Returns: (audio_features, hard_label, soft_label)")
print(f"   - Uso: Knowledge Distillation training")

# Validation dataset: CON soft labels
val_dataset_kd = NoisyStudentDatasetWithSoftLabels(
    iemocap_dataset=val_iemocap_dataset,
    soft_labels_dict=val_soft_labels,
    hard_labels_list=val_hard_labels
)

print(f"\n✅ Validation dataset (with soft labels):")
print(f"   - Size: {len(val_dataset_kd)}")
print(f"   - Returns: (audio_features, hard_label, soft_label)")
print(f"   - Uso: Validation durante KD training")

# Test dataset: SENZA soft labels (PURO)
test_dataset_kd = NoisyStudentDatasetNoSoftLabels(
    iemocap_dataset=test_iemocap_dataset,
    hard_labels_list=test_hard_labels
)

print(f"\n✅ Test dataset (NO soft labels - PURE):")
print(f"   - Size: {len(test_dataset_kd)}")
print(f"   - Returns: (audio_features, hard_label)")
print(f"   - Uso: Final evaluation (uncontaminated by teacher)")

# ---- CREA I DATALOADER FINALI ----
print("\n" + "="*80)
print("📦 CREATING DATALOADERS FOR NOISY STUDENT TRAINING")
print("="*80)

train_dataloader_kd = DataLoader(
    train_dataset_kd,
    batch_size=Config.BATCH_SIZE_STUDENT_IEMOCAP,
    shuffle=True,  # Ora possiamo permetterci di shufflare (soft labels sono deterministici)
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

val_dataloader_kd = DataLoader(
    val_dataset_kd,
    batch_size=Config.BATCH_SIZE_STUDENT_IEMOCAP,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

test_dataloader_kd = DataLoader(
    test_dataset_kd,
    batch_size=Config.BATCH_SIZE_STUDENT_IEMOCAP,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✅ Train DataLoader: batch_size={Config.BATCH_SIZE_STUDENT_IEMOCAP}, shuffle=True")
print(f"✅ Val DataLoader:   batch_size={Config.BATCH_SIZE_STUDENT_IEMOCAP}, shuffle=False")
print(f"✅ Test DataLoader:  batch_size={Config.BATCH_SIZE_STUDENT_IEMOCAP}, shuffle=False")

# ---- VERIFICA: INSPECT UN BATCH ----
print("\n" + "="*80)
print("🔍 SAMPLE BATCH INSPECTION")
print("="*80)

sample_batch_train = next(iter(train_dataloader_kd))
print(f"\n📊 Train batch keys: {sample_batch_train.keys()}")
print(f"   - audio_features shape: {sample_batch_train['audio_features'].shape}")
print(f"   - hard_label shape: {sample_batch_train['hard_label'].shape}")
print(f"   - soft_label shape: {sample_batch_train['soft_label'].shape}")

sample_batch_val = next(iter(val_dataloader_kd))
print(f"\n📊 Val batch keys: {sample_batch_val.keys()}")
print(f"   - audio_features shape: {sample_batch_val['audio_features'].shape}")
print(f"   - hard_label shape: {sample_batch_val['hard_label'].shape}")
print(f"   - soft_label shape: {sample_batch_val['soft_label'].shape}")

sample_batch_test = next(iter(test_dataloader_kd))
print(f"\n📊 Test batch keys: {sample_batch_test.keys()}")
print(f"   - audio_features shape: {sample_batch_test['audio_features'].shape}")
print(f"   - hard_label shape: {sample_batch_test['hard_label'].shape}")
print(f"   - ⚠️ NO soft_label (test set remains pure)")

# ---- STATISTICHE FINALI ----
print("\n" + "="*80)
print("📈 DATASETS SUMMARY")
print("="*80)
print(f"\n✅ Train split:      {len(train_dataloader_kd)} batches × {Config.BATCH_SIZE_STUDENT_IEMOCAP} = {len(train_dataset_kd)} samples")
print(f"✅ Val split:        {len(val_dataloader_kd)} batches × {Config.BATCH_SIZE_STUDENT_IEMOCAP} = {len(val_dataset_kd)} samples")
print(f"✅ Test split:       {len(test_dataloader_kd)} batches × {Config.BATCH_SIZE_STUDENT_IEMOCAP} = {len(test_dataset_kd)} samples")

print(f"\n📝 DATASET CONFIGURATION:")
print(f"   Train: WITH soft labels (from RAVDESS teacher)")
print(f"   Val:   WITH soft labels (from RAVDESS teacher)")
print(f"   Test:  WITHOUT soft labels (PURE IEMOCAP ground truth)")

print("\n" + "="*80)
print("✅ CUSTOM DATASETS READY FOR NOISY STUDENT TRAINING")
print("="*80 + "\n")


🔧 CREATING CUSTOM DATASET WRAPPERS FOR NOISY STUDENT

✅ Train dataset (with soft labels):
   - Size: 1678
   - Returns: (audio_features, hard_label, soft_label)
   - Uso: Knowledge Distillation training

✅ Validation dataset (with soft labels):
   - Size: 534
   - Returns: (audio_features, hard_label, soft_label)
   - Uso: Validation durante KD training

✅ Test dataset (NO soft labels - PURE):
   - Size: 731
   - Returns: (audio_features, hard_label)
   - Uso: Final evaluation (uncontaminated by teacher)

📦 CREATING DATALOADERS FOR NOISY STUDENT TRAINING

✅ Train DataLoader: batch_size=64, shuffle=True
✅ Val DataLoader:   batch_size=64, shuffle=False
✅ Test DataLoader:  batch_size=64, shuffle=False

🔍 SAMPLE BATCH INSPECTION

📊 Train batch keys: dict_keys(['audio_features', 'hard_label', 'soft_label', 'emotion_id'])
   - audio_features shape: torch.Size([64, 1, 128, 94])
   - hard_label shape: torch.Size([64])
   - soft_label shape: torch.Size([64, 4])

📊 Val batch keys: dict_keys(['a

In [10]:
# ==============================================================================
# SEZIONE 5: KNOWLEDGE DISTILLATION LOSS FUNCTION
# ==============================================================================
# Implementa la funzione di loss per Knowledge Distillation:
#   Loss_total = α_CE * CrossEntropy(student, hard_labels) 
#              + α_KL * KL_divergence(student_soft, teacher_soft)
#
# Questa combinazione:
# - Mantiene l'allineamento con ground truth (hard labels)
# - Inoltre insegna allo student i "soft" insegnamenti del teacher
# ==============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- FUNZIONE PER CALCOLARE KNOWLEDGE DISTILLATION LOSS ----
def compute_kd_loss(student_logits, teacher_logits, hard_labels, 
                    temperature=Config.TEMPERATURE, 
                    alpha_ce=Config.ALPHA_CE, 
                    alpha_kl=Config.ALPHA_KL,
                    device='cpu'):
    """
    Calcola la combined loss per Knowledge Distillation.
    
    Args:
        student_logits: Output del modello student shape=(batch_size, num_classes)
        teacher_logits: Output del modello teacher shape=(batch_size, num_classes)
        hard_labels: Ground truth labels shape=(batch_size,)
        temperature: Temperature scaling per soft targets
        alpha_ce: Peso sulla CrossEntropy loss (ground truth)
        alpha_kl: Peso sulla KL divergence loss (teacher guidance)
        device: Device per il calcolo
    
    Returns:
        loss_dict: Dict con {'total_loss': float, 'loss_ce': float, 'loss_kl': float}
    """
    
    # ---- COMPONENTE 1: CROSS ENTROPY LOSS (hard labels) ----
    # Penalizza quando student non predice correttamente il ground truth
    criterion_ce = nn.CrossEntropyLoss()
    loss_ce = criterion_ce(student_logits, hard_labels)
    
    # ---- COMPONENTE 2: KL DIVERGENCE LOSS (soft targets) ----
    # Penalizza quando soft predictions dello student non concordano con teacher
    
    # Calcola soft probabilities dal teacher (con temperature scaling)
    # Temperature alta → distribuzioni più smooth → gradients più soft
    teacher_soft = F.softmax(teacher_logits / temperature, dim=1)
    
    # Calcola log-soft probabilities dallo student
    # usa log_softmax per numerica stability
    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    
    # KL divergence: somma su tutte le classi
    # Note: KL è asimmetrica - misura quanto student diverge da teacher
    loss_kl = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean')
    
    # ---- COMBINAZIONE PESATA ----
    loss_total = alpha_ce * loss_ce + alpha_kl * loss_kl
    
    return {
        'total_loss': loss_total,
        'loss_ce': loss_ce.item(),
        'loss_kl': loss_kl.item(),
        'weighted_ce': (alpha_ce * loss_ce).item(),
        'weighted_kl': (alpha_kl * loss_kl).item()
    }


# ---- FUNZIONE ALTERNATIVA CON SOFT LABELS (se disponibili come targets) ----
def compute_kd_loss_with_soft_targets(student_logits, soft_targets, hard_labels,
                                      temperature=Config.TEMPERATURE,
                                      alpha_ce=Config.ALPHA_CE,
                                      alpha_kl=Config.ALPHA_KL):
    """
    Variante quando i soft targets sono già disponibili (come in cella 4).
    
    Args:
        student_logits: Output student shape=(batch_size, num_classes)
        soft_targets: Pre-computed soft labels shape=(batch_size, num_classes)
        hard_labels: Ground truth shape=(batch_size,)
        temperature: Temperature scaling
        alpha_ce: Peso CE
        alpha_kl: Peso KL
    
    Returns:
        loss_dict: Dict con losses
    """
    
    # ---- CROSS ENTROPY vs HARD LABELS ----
    criterion_ce = nn.CrossEntropyLoss()
    loss_ce = criterion_ce(student_logits, hard_labels)
    
    # ---- KL DIVERGENCE vs SOFT TARGETS ----
    # I soft targets sono già probabilità (da cella 3)
    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    loss_kl = F.kl_div(student_log_soft, soft_targets, reduction='batchmean')
    
    # ---- COMBINAZIONE ----
    loss_total = alpha_ce * loss_ce + alpha_kl * loss_kl
    
    return {
        'total_loss': loss_total,
        'loss_ce': loss_ce.item(),
        'loss_kl': loss_kl.item(),
        'weighted_ce': (alpha_ce * loss_ce).item(),
        'weighted_kl': (alpha_kl * loss_kl).item()
    }


print("\n" + "="*80)
print("📋 KNOWLEDGE DISTILLATION LOSS FUNCTIONS DEFINED")
print("="*80)
print(f"\n🎯 Loss Configuration:")
print(f"   - TEMPERATURE: {Config.TEMPERATURE}")
print(f"   - ALPHA_CE (hard labels weight): {Config.ALPHA_CE}")
print(f"   - ALPHA_KL (soft targets weight): {Config.ALPHA_KL}")
print(f"   - Sum: {Config.ALPHA_CE + Config.ALPHA_KL}")

print(f"\n📌 Loss Formula:")
print(f"   Loss = {Config.ALPHA_CE} × CrossEntropy(student, hard_labels)")
print(f"        + {Config.ALPHA_KL} × KL_Divergence(student_soft, teacher_soft)")
print(f"\n   Interpretation:")
print(f"   - CE component (α={Config.ALPHA_CE}): Student deve predire ground truth")
print(f"   - KL component (α={Config.ALPHA_KL}): Student deve imitare teacher")

# ---- TEST: VERIFICA CON UN BATCH ----
print("\n" + "="*80)
print("✅ TESTING KD LOSS COMPUTATION")
print("="*80)

# Prendi un batch di training data
test_batch = next(iter(train_dataloader_kd))

print(f"\n📊 Input batch:")
print(f"   - audio_features shape: {test_batch['audio_features'].shape}")
print(f"   - hard_label shape: {test_batch['hard_label'].shape}")
print(f"   - soft_label shape: {test_batch['soft_label'].shape}")

# Simula output dello student (random, non è allenato ancora)
batch_size = test_batch['audio_features'].shape[0]
num_classes = 4  # Neutral, Happy, Sad, Angry
dummy_student_logits = torch.randn(batch_size, num_classes, device=device)

# Simula output del teacher (è deterministico, da inference)
# In realtà useremo i logit salvati o li ricalcoleremo on-the-fly
dummy_teacher_logits = torch.randn(batch_size, num_classes, device=device)

# Calcola loss con teacher logits
loss_dict_1 = compute_kd_loss(
    student_logits=dummy_student_logits,
    teacher_logits=dummy_teacher_logits,
    hard_labels=test_batch['hard_label'].to(device),
    temperature=Config.TEMPERATURE,
    alpha_ce=Config.ALPHA_CE,
    alpha_kl=Config.ALPHA_KL,
    device=device
)

print(f"\n🔴 Loss with teacher logits (Metodo 1 - durante training):")
print(f"   - Loss CE:          {loss_dict_1['loss_ce']:.6f}")
print(f"   - Loss KL:          {loss_dict_1['loss_kl']:.6f}")
print(f"   - Weighted CE:      {loss_dict_1['weighted_ce']:.6f}")
print(f"   - Weighted KL:      {loss_dict_1['weighted_kl']:.6f}")
print(f"   - TOTAL LOSS:       {loss_dict_1['total_loss'].item():.6f}")

# Calcola loss con soft targets pre-calcolati (da cella 3)
loss_dict_2 = compute_kd_loss_with_soft_targets(
    student_logits=dummy_student_logits,
    soft_targets=test_batch['soft_label'].to(device),
    hard_labels=test_batch['hard_label'].to(device),
    temperature=Config.TEMPERATURE,
    alpha_ce=Config.ALPHA_CE,
    alpha_kl=Config.ALPHA_KL
)

print(f"\n🟢 Loss with soft targets (Metodo 2 - usa soft labels salvati):")
print(f"   - Loss CE:          {loss_dict_2['loss_ce']:.6f}")
print(f"   - Loss KL:          {loss_dict_2['loss_kl']:.6f}")
print(f"   - Weighted CE:      {loss_dict_2['weighted_ce']:.6f}")
print(f"   - Weighted KL:      {loss_dict_2['weighted_kl']:.6f}")
print(f"   - TOTAL LOSS:       {loss_dict_2['total_loss'].item():.6f}")

print(f"\n💡 WHICH METHOD?")
print(f"   ✅ METODO 2 (soft_targets) = PREFERITO durante training")
print(f"      Reason: Soft targets sono già salvati da cella 3")
print(f"              Riducono computation (no need to run teacher)")
print(f"              Deterministic (garantisce reproducibility)")
print(f"\n   🔹 METODO 1 (teacher_logits) = Alternativa flessibile")
print(f"      Use: Se ricalcoli i logit on-the-fly")

print("\n" + "="*80)
print("✅ KNOWLEDGE DISTILLATION LOSS READY")
print("="*80 + "\n")


📋 KNOWLEDGE DISTILLATION LOSS FUNCTIONS DEFINED

🎯 Loss Configuration:
   - TEMPERATURE: 4.0
   - ALPHA_CE (hard labels weight): 0.7
   - ALPHA_KL (soft targets weight): 0.3
   - Sum: 1.0

📌 Loss Formula:
   Loss = 0.7 × CrossEntropy(student, hard_labels)
        + 0.3 × KL_Divergence(student_soft, teacher_soft)

   Interpretation:
   - CE component (α=0.7): Student deve predire ground truth
   - KL component (α=0.3): Student deve imitare teacher

✅ TESTING KD LOSS COMPUTATION

📊 Input batch:
   - audio_features shape: torch.Size([64, 1, 128, 94])
   - hard_label shape: torch.Size([64])
   - soft_label shape: torch.Size([64, 4])

🔴 Loss with teacher logits (Metodo 1 - durante training):
   - Loss CE:          1.814322
   - Loss KL:          0.043889
   - Weighted CE:      1.270025
   - Weighted KL:      0.013167
   - TOTAL LOSS:       1.283192

🟢 Loss with soft targets (Metodo 2 - usa soft labels salvati):
   - Loss CE:          1.814322
   - Loss KL:          0.109981
   - Weighted C

In [ ]:
# ==============================================================================
# SEZIONE 6: STUDENT TRAINING LOOP WITH KNOWLEDGE DISTILLATION
# ==============================================================================
# Allena il modello student usando il Knowledge Distillation Loss
# - Student impara dai soft targets del teacher
# - Mantiene l'allineamento con ground truth (hard labels)
# ==============================================================================

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time
from pathlib import Path

# ---- INIZIALIZZAZIONE STUDENT MODEL ----
print("\n" + "="*80)
print("🎓 INITIALIZING STUDENT MODEL")
print("="*80)

# Crea nuovo student model (architettura identica al teacher, ma PESI casuali)
student_model = get_model(
    model_name='CRNN_BiLSTM',
    batch_size=Config.BATCH_SIZE_STUDENT_IEMOCAP,
    time_steps=Config.TIME_STEPS_IEMOCAP,
    dropout=Config.DROPOUT_STUDENT_IEMOCAP  # Dropout più alto per student
).to(device)

print(f"✅ Student Model Initialized")
print(f"   - Architecture: CRNN_BiLSTM")
print(f"   - Batch size: {Config.BATCH_SIZE_STUDENT_IEMOCAP}")
print(f"   - Time steps: {Config.TIME_STEPS_IEMOCAP}")
print(f"   - Dropout: {Config.DROPOUT_STUDENT_IEMOCAP}")
print(f"   - Device: {device}")
print(f"   - Total Parameters: {sum(p.numel() for p in student_model.parameters()):,}")

# ---- SETUP OPTIMIZER & SCHEDULER ----
print(f"\n📊 Setting up Optimizer & Scheduler")

# Optimizer: Adam con learning rate e weight decay
optimizer = optim.Adam(
    student_model.parameters(),
    lr=Config.LEARNING_RATE_STUDENT_IEMOCAP,
    weight_decay=Config.WEIGHT_DECAY_STUDENT_IEMOCAP
)

# Scheduler: Riduci LR se validation loss non migliora
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,  # Moltiplicare LR per 0.5
    patience=3,   # Aspetta 3 epoch senza miglioramento
    verbose=True
)

print(f"   ✅ Optimizer: Adam")
print(f"      - Learning rate: {Config.LEARNING_RATE_STUDENT_IEMOCAP}")
print(f"      - Weight decay: {Config.WEIGHT_DECAY_STUDENT_IEMOCAP}")
print(f"   ✅ Scheduler: ReduceLROnPlateau")
print(f"      - Factor: 0.5 (riduce LR del 50%)")
print(f"      - Patience: 3 epochs")

# ---- SETUP CHECKPOINT DIRECTORY ----
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "iemocap_only" / "noisy_student_kd"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"   ✅ Checkpoint directory: {checkpoint_dir}")

# ---- TRAINING PARAMETERS ----
num_epochs = Config.NUM_EPOCHS_STUDENT_IEMOCAP
early_stopping_patience = Config.EARLY_STOPPING_PATIENCE_STUDENT
best_val_loss = float('inf')
patience_counter = 0

print(f"\n⚙️ Training Configuration:")
print(f"   - Total epochs: {num_epochs}")
print(f"   - Early stopping patience: {early_stopping_patience} epochs")
print(f"   - Train batches per epoch: {len(train_dataloader_kd)}")
print(f"   - Val batches per epoch: {len(val_dataloader_kd)}")

# ---- TRAINING LOOP ----
print("\n" + "="*80)
print("🚀 STARTING STUDENT TRAINING WITH KNOWLEDGE DISTILLATION")
print("="*80)

training_history = {
    'train_loss': [],
    'train_loss_ce': [],
    'train_loss_kl': [],
    'val_loss': [],
    'val_loss_ce': [],
    'val_loss_kl': [],
    'learning_rates': []
}

for epoch in range(num_epochs):
    # ---- TRAINING PHASE ----
    student_model.train()  # Enable dropout
    
    epoch_train_loss = 0.0
    epoch_train_loss_ce = 0.0
    epoch_train_loss_kl = 0.0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_dataloader_kd):
        # Forward pass
        audio_features = batch['audio_features'].to(device)
        hard_labels = batch['hard_label'].to(device)
        soft_labels = batch['soft_label'].to(device)
        
        # Student forward pass
        student_logits = student_model(audio_features)
        
        # Calcola KD loss con soft targets pre-calcolati
        loss_dict = compute_kd_loss_with_soft_targets(
            student_logits=student_logits,
            soft_targets=soft_labels,
            hard_labels=hard_labels,
            temperature=Config.TEMPERATURE,
            alpha_ce=Config.ALPHA_CE,
            alpha_kl=Config.ALPHA_KL
        )
        
        total_loss = loss_dict['total_loss']
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        # Accumula losses
        epoch_train_loss += loss_dict['total_loss'].item()
        epoch_train_loss_ce += loss_dict['loss_ce']
        epoch_train_loss_kl += loss_dict['loss_kl']
        num_batches += 1
        
        # Progress
        if (batch_idx + 1) % 10 == 0:
            print(f"   Epoch {epoch+1}/{num_epochs} - Batch {batch_idx+1}/{len(train_dataloader_kd)}: " + 
                  f"Loss={loss_dict['total_loss'].item():.4f} (CE={loss_dict['loss_ce']:.4f}, KL={loss_dict['loss_kl']:.4f})")
    
    # Average training loss
    avg_train_loss = epoch_train_loss / num_batches
    avg_train_loss_ce = epoch_train_loss_ce / num_batches
    avg_train_loss_kl = epoch_train_loss_kl / num_batches
    
    # ---- VALIDATION PHASE ----
    student_model.eval()  # Disable dropout
    
    epoch_val_loss = 0.0
    epoch_val_loss_ce = 0.0
    epoch_val_loss_kl = 0.0
    num_val_batches = 0
    
    with torch.no_grad():
        for batch in val_dataloader_kd:
            audio_features = batch['audio_features'].to(device)
            hard_labels = batch['hard_label'].to(device)
            soft_labels = batch['soft_label'].to(device)
            
            student_logits = student_model(audio_features)
            
            loss_dict = compute_kd_loss_with_soft_targets(
                student_logits=student_logits,
                soft_targets=soft_labels,
                hard_labels=hard_labels,
                temperature=Config.TEMPERATURE,
                alpha_ce=Config.ALPHA_CE,
                alpha_kl=Config.ALPHA_KL
            )
            
            epoch_val_loss += loss_dict['total_loss'].item()
            epoch_val_loss_ce += loss_dict['loss_ce']
            epoch_val_loss_kl += loss_dict['loss_kl']
            num_val_batches += 1
    
    avg_val_loss = epoch_val_loss / num_val_batches
    avg_val_loss_ce = epoch_val_loss_ce / num_val_batches
    avg_val_loss_kl = epoch_val_loss_kl / num_val_batches
    
    # Salva history
    training_history['train_loss'].append(avg_train_loss)
    training_history['train_loss_ce'].append(avg_train_loss_ce)
    training_history['train_loss_kl'].append(avg_train_loss_kl)
    training_history['val_loss'].append(avg_val_loss)
    training_history['val_loss_ce'].append(avg_val_loss_ce)
    training_history['val_loss_kl'].append(avg_val_loss_kl)
    training_history['learning_rates'].append(optimizer.param_groups[0]['lr'])
    
    # ---- SCHEDULER STEP ----
    scheduler.step(avg_val_loss)
    
    # ---- CHECKPOINT SAVING ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Salva best model
        checkpoint_path = checkpoint_dir / "best_model_kd.pth"
        torch.save(student_model.state_dict(), checkpoint_path)
        
        print(f"\n✅ EPOCH {epoch+1} - NEW BEST MODEL SAVED!")
        print(f"   Val Loss: {avg_val_loss:.6f} (CE: {avg_val_loss_ce:.6f}, KL: {avg_val_loss_kl:.6f})")
    else:
        patience_counter += 1
        print(f"\n⏳ EPOCH {epoch+1} - No improvement")
        print(f"   Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        print(f"   Patience: {patience_counter}/{early_stopping_patience}")
    
    # ---- EARLY STOPPING ----
    if patience_counter >= early_stopping_patience:
        print(f"\n🛑 EARLY STOPPING at epoch {epoch+1}")
        print(f"   Best val loss: {best_val_loss:.6f}")
        break

print("\n" + "="*80)
print("✅ STUDENT TRAINING COMPLETED")
print("="*80)
print(f"\n📊 Training Summary:")
print(f"   - Total epochs trained: {epoch+1}/{num_epochs}")
print(f"   - Best validation loss: {best_val_loss:.6f}")
print(f"   - Best model saved at: {checkpoint_dir / 'best_model_kd.pth'}")

# ---- CARICA IL MIGLIOR MODELLO ----
student_model.load_state_dict(torch.load(checkpoint_dir / "best_model_kd.pth", map_location=device))
print(f"\n✅ Best model loaded for evaluation")


In [ ]:

# ---- VISUALIZZAZIONE TRAINING HISTORY ----
print("\n" + "="*80)
print("📊 PLOTTING TRAINING HISTORY")
print("="*80)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Crea figura con 4 subplot
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

epochs_range = range(1, len(training_history['train_loss']) + 1)

# ---- 1. COMBINED LOSS (Total, CE, KL) ----
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_range, training_history['train_loss'], 'b-', linewidth=2, label='Train Total Loss', marker='o', markersize=4)
ax1.plot(epochs_range, training_history['val_loss'], 'r-', linewidth=2, label='Val Total Loss', marker='s', markersize=4)
ax1.fill_between(epochs_range, training_history['train_loss'], alpha=0.2, color='blue')
ax1.fill_between(epochs_range, training_history['val_loss'], alpha=0.2, color='red')
ax1.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax1.set_ylabel('Loss', fontsize=11, fontweight='bold')
ax1.set_title('Total Loss (KD: CE + KL)', fontsize=12, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# ---- 2. CE LOSS vs KL LOSS (Training) ----
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_range, training_history['train_loss_ce'], 'orange', linewidth=2, label='Train CE Loss', marker='o', markersize=4)
ax2.plot(epochs_range, training_history['train_loss_kl'], 'green', linewidth=2, label='Train KL Loss', marker='^', markersize=4)
ax2.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax2.set_ylabel('Loss Component', fontsize=11, fontweight='bold')
ax2.set_title('Training Loss Components (CE vs KL)', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

# ---- 3. VALIDATION LOSS COMPONENTS ----
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(epochs_range, training_history['val_loss_ce'], 'red', linewidth=2, label='Val CE Loss', marker='o', markersize=4)
ax3.plot(epochs_range, training_history['val_loss_kl'], 'purple', linewidth=2, label='Val KL Loss', marker='^', markersize=4)
ax3.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax3.set_ylabel('Loss Component', fontsize=11, fontweight='bold')
ax3.set_title('Validation Loss Components (CE vs KL)', fontsize=12, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)

# ---- 4. LEARNING RATE PROGRESSION ----
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(epochs_range, training_history['learning_rates'], 'darkblue', linewidth=2.5, marker='D', markersize=5)
ax4.fill_between(epochs_range, training_history['learning_rates'], alpha=0.2, color='darkblue')
ax4.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax4.set_ylabel('Learning Rate', fontsize=11, fontweight='bold')
ax4.set_title('Learning Rate Schedule (ReduceLROnPlateau)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, which='both')
ax4.set_yscale('log')

plt.suptitle('Knowledge Distillation Training History - IEMOCAP Student', 
             fontsize=14, fontweight='bold', y=0.995)

# Salva figura
training_plots_dir = checkpoint_dir / "training_plots"
training_plots_dir.mkdir(parents=True, exist_ok=True)
training_history_path = training_plots_dir / "training_history.png"
plt.savefig(training_history_path, dpi=150, bbox_inches='tight')
print(f"✅ Training history plot saved to: {training_history_path}")
plt.close()

# ---- STAMPA STATISTICHE TRAINING ----
print(f"\n📈 TRAINING STATISTICS:")
print(f"   Min Train Loss: {min(training_history['train_loss']):.6f} @ epoch {np.argmin(training_history['train_loss']) + 1}")
print(f"   Min Val Loss:   {min(training_history['val_loss']):.6f} @ epoch {np.argmin(training_history['val_loss']) + 1}")
print(f"   Final Train Loss: {training_history['train_loss'][-1]:.6f}")
print(f"   Final Val Loss:   {training_history['val_loss'][-1]:.6f}")
print(f"   Final Learning Rate: {training_history['learning_rates'][-1]:.2e}")


In [ ]:
# ==============================================================================
# SEZIONE 7: EVALUATION ON TEST SET
# ==============================================================================
# Valuta il modello student sul test set PURO (senza soft labels del teacher)
# Calcola metriche di performance: accuracy, precision, recall, F1, confusion matrix
# ==============================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*80)
print("📊 EVALUATING STUDENT MODEL ON TEST SET")
print("="*80)

# ---- CARICA IL MIGLIOR MODELLO (se non già caricato) ----
student_model.eval()

# ---- INFERENCE SU TEST SET ----
print("\n🔍 Running inference on test set...")

all_predictions = []
all_hard_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_dataloader_kd):
        audio_features = batch['audio_features'].to(device)
        hard_labels = batch['hard_label'].cpu().numpy()
        
        # Forward pass
        student_logits = student_model(audio_features)
        
        # Predizioni: argmax dei logit
        predictions = torch.argmax(student_logits, dim=1).cpu().numpy()
        
        all_predictions.extend(predictions)
        all_hard_labels.extend(hard_labels)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"   Processed {batch_idx + 1}/{len(test_dataloader_kd)} batches")

all_predictions = np.array(all_predictions)
all_hard_labels = np.array(all_hard_labels)

print(f"\n✅ Inference completed")
print(f"   - Total test samples: {len(all_hard_labels)}")

# ---- METRICHE DI PERFORMANCE ----
print("\n" + "="*80)
print("📈 TEST SET METRICS")
print("="*80)

# Accuracy globale
accuracy = accuracy_score(all_hard_labels, all_predictions)
print(f"\n🎯 OVERALL ACCURACY: {accuracy*100:.2f}%")

# Precision, Recall, F1 per classe
emotion_names = ['Neutral', 'Happy', 'Sad', 'Angry']

precision = precision_score(all_hard_labels, all_predictions, average='weighted', zero_division=0)
recall = recall_score(all_hard_labels, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(all_hard_labels, all_predictions, average='weighted', zero_division=0)

print(f"\n📊 WEIGHTED METRICS:")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall:    {recall:.4f}")
print(f"   - F1-Score:  {f1:.4f}")

# Per classe
print(f"\n📋 PER-CLASS METRICS:")
print(f"\n{classification_report(all_hard_labels, all_predictions, target_names=emotion_names, zero_division=0)}")

# ---- CONFUSION MATRIX ----
print(f"\n" + "="*80)
print("🔥 CONFUSION MATRIX")
print("="*80)

cm = confusion_matrix(all_hard_labels, all_predictions)
print(f"\nConfusion Matrix:")
print(cm)

# Visualizzazione
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotion_names, yticklabels=emotion_names,
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Student Model Confusion Matrix on Test Set')
plt.tight_layout()

# Salva figura
eval_plots_dir = checkpoint_dir / "eval_plots"
eval_plots_dir.mkdir(parents=True, exist_ok=True)
confusion_matrix_path = eval_plots_dir / "confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=100, bbox_inches='tight')
print(f"\n✅ Confusion matrix saved to: {confusion_matrix_path}")
plt.close()

# ---- ERRORE PER CLASSE ----
print(f"\n" + "="*80)
print("❌ ERROR ANALYSIS")
print("="*80)

for class_idx, class_name in enumerate(emotion_names):
    class_mask = all_hard_labels == class_idx
    if class_mask.sum() > 0:
        class_accuracy = accuracy_score(
            all_hard_labels[class_mask], 
            all_predictions[class_mask]
        )
        print(f"\n{class_name:10s}: {class_accuracy*100:6.2f}% accuracy ({class_mask.sum()} samples)")

# ---- SALVA RISULTATI ----
print(f"\n" + "="*80)
print("💾 SAVING EVALUATION RESULTS")
print("="*80)

results_path = checkpoint_dir / "test_results.txt"
with open(results_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("NOISY STUDENT KNOWLEDGE DISTILLATION - TEST SET EVALUATION\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"Overall Accuracy: {accuracy*100:.2f}%\n")
    f.write(f"Weighted Precision: {precision:.4f}\n")
    f.write(f"Weighted Recall: {recall:.4f}\n")
    f.write(f"Weighted F1-Score: {f1:.4f}\n\n")
    
    f.write("Per-Class Metrics:\n")
    f.write(classification_report(all_hard_labels, all_predictions, target_names=emotion_names, zero_division=0))
    
    f.write("\n\nConfusion Matrix:\n")
    for i, row in enumerate(cm):
        f.write(f"{emotion_names[i]:10s}: {row}\n")

print(f"✅ Results saved to: {results_path}")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETED")
print("="*80 + "\n")


In [ ]:
# ==============================================================================
# SEZIONE 8: ITERATIVE NOISY STUDENT (OPTIONAL)
# ==============================================================================
# Implementazione dell'approccio iterativo di Noisy Student:
# 1. Student (T_i) allena su soft targets da Teacher (T_{i-1})
# 2. Studente migliora e diventa nuovo Teacher (T_{i+1})
# 3. Genera nuovi soft labels con noise aggiunto
# 4. Repeat per NUM_ITERATIONS_NOISY_STUDENT volte
#
# Questo permette di creare un ciclo di miglioramento iterativo.
# ==============================================================================

print("\n" + "="*80)
print("🔄 ITERATIVE NOISY STUDENT TRAINING")
print("="*80)

num_iterations = Config.NUM_ITERATIONS_NOISY_STUDENT

print(f"\n⚙️ Configuration:")
print(f"   - Total iterations: {num_iterations}")
print(f"   - Noise variance (label smoothing): {Config.LABEL_SMOOTHING}")

if num_iterations <= 1:
    print(f"\n⚠️ Skipping iterative training (num_iterations={num_iterations})")
    print(f"   To use iterative training, set NUM_ITERATIONS_NOISY_STUDENT > 1 in config.py")
    
else:
    print(f"\n" + "="*80)
    print("📚 ITERATION DETAILS")
    print("="*80)
    
    iteration_history = {
        'iteration': [],
        'test_accuracy': [],
        'test_f1': [],
        'teacher_accuracy': [],
        'soft_label_alignment': []
    }
    
    for iteration in range(1, num_iterations):
        print(f"\n\n{'='*80}")
        print(f"🔄 ITERATION {iteration}")
        print(f"{'='*80}")
        
        # ---- SALVA STUDENT COME NUOVO TEACHER ----
        print(f"\n✅ Saving student as new teacher (T_{iteration})")
        
        teacher_iteration_path = checkpoint_dir / f"teacher_iteration_{iteration}.pth"
        torch.save(student_model.state_dict(), teacher_iteration_path)
        
        # Dopo l'iterazione 1, il nuovo teacher è lo student appena allenato
        old_teacher_model = student_model
        
        # ---- GENERA NUOVI SOFT LABELS CON NOISE ----
        print(f"\n🔄 Generating soft labels for iteration {iteration} (with noise)")
        
        # Con label smoothing (smoothing dei soft targets)
        noise_variance = Config.LABEL_SMOOTHING
        
        # Funzione per aggiungere noise ai soft labels
        def add_label_smoothing(soft_probs, smoothing_factor=0.1):
            """
            Aggiunge label smoothing ai soft targets.
            Smoothing = uniform distribution mix-in
            
            Args:
                soft_probs: array di probabilità shape=(num_classes,)
                smoothing_factor: quanto mixare con uniform distribution
            
            Returns:
                smoothed_probs: probabilità smooothed
            """
            num_classes = len(soft_probs)
            uniform_dist = np.ones(num_classes) / num_classes
            smoothed = (1 - smoothing_factor) * soft_probs + smoothing_factor * uniform_dist
            return smoothed / smoothed.sum()  # Rinormalizza per somma = 1
        
        # Genera soft labels dalla nuova teacher
        old_teacher_model.eval()
        
        print(f"\n   Train split (re-generating with noise):")
        train_soft_labels_iter = {}
        with torch.no_grad():
            for batch_idx, batch in enumerate(train_iemocap_loader):
                audio_features = batch['audio_features'].to(device)
                emotion_ids = batch['emotion_id'].to(device)
                
                logits = old_teacher_model(audio_features)
                soft_probs = F.softmax(logits / Config.TEMPERATURE, dim=1)  # Temperature scaling
                
                for i in range(audio_features.shape[0]):
                    sample_idx = batch_idx * Config.BATCH_SIZE_IEMOCAP + i
                    # Aggiungi smoothing al soft label
                    smoothed_soft = add_label_smoothing(
                        soft_probs[i].cpu().numpy(), 
                        smoothing_factor=noise_variance
                    )
                    train_soft_labels_iter[sample_idx] = smoothed_soft
                
                if (batch_idx + 1) % 10 == 0:
                    print(f"      Processed {batch_idx * Config.BATCH_SIZE_IEMOCAP} samples")
        
        print(f"   ✅ Generated {len(train_soft_labels_iter)} smoothed soft labels")
        
        # ---- CREA NUOVO STUDENT ----
        print(f"\n🎓 Initializing new student model for iteration {iteration}")
        
        student_model_iter = get_model(
            model_name='CRNN_BiLSTM',
            batch_size=Config.BATCH_SIZE_STUDENT_IEMOCAP,
            time_steps=Config.TIME_STEPS_IEMOCAP,
            dropout=Config.DROPOUT_STUDENT_IEMOCAP
        ).to(device)
        
        print(f"   ✅ New student initialized (fresh weights)")
        
        # ---- ALLENA NUOVO STUDENT ----
        print(f"\n🚀 Training student for iteration {iteration}")
        
        # Setup ottimizzatore e scheduler
        optimizer_iter = optim.Adam(
            student_model_iter.parameters(),
            lr=Config.LEARNING_RATE_STUDENT_IEMOCAP,
            weight_decay=Config.WEIGHT_DECAY_STUDENT_IEMOCAP
        )
        
        scheduler_iter = ReduceLROnPlateau(
            optimizer_iter, mode='min', factor=0.5, patience=3, verbose=False
        )
        
        checkpoint_dir_iter = checkpoint_dir / f"iteration_{iteration}"
        checkpoint_dir_iter.mkdir(parents=True, exist_ok=True)
        
        best_val_loss_iter = float('inf')
        patience_counter_iter = 0
        
        # Training loop ridotto (fewer epochs per iterazioni)
        num_epochs_iter = min(Config.NUM_EPOCHS_STUDENT_IEMOCAP // 2, 40)
        
        for epoch in range(num_epochs_iter):
            student_model_iter.train()
            epoch_loss = 0.0
            num_batches = 0
            
            for batch in train_dataloader_kd:
                audio_features = batch['audio_features'].to(device)
                hard_labels = batch['hard_label'].to(device)
                sample_indices = range(num_batches * Config.BATCH_SIZE_STUDENT_IEMOCAP,
                                     (num_batches + 1) * Config.BATCH_SIZE_STUDENT_IEMOCAP)
                
                # Usa soft labels dall'iterazione corrente
                soft_labels_batch = torch.tensor(
                    np.array([train_soft_labels_iter.get(idx, batch['soft_label'][i].numpy()) 
                             for i, idx in enumerate(sample_indices)]),
                    dtype=torch.float32,
                    device=device
                )
                
                student_logits = student_model_iter(audio_features)
                
                loss_dict = compute_kd_loss_with_soft_targets(
                    student_logits=student_logits,
                    soft_targets=soft_labels_batch,
                    hard_labels=hard_labels,
                    temperature=Config.TEMPERATURE,
                    alpha_ce=Config.ALPHA_CE,
                    alpha_kl=Config.ALPHA_KL
                )
                
                optimizer_iter.zero_grad()
                loss_dict['total_loss'].backward()
                optimizer_iter.step()
                
                epoch_loss += loss_dict['total_loss'].item()
                num_batches += 1
            
            # Validation
            student_model_iter.eval()
            epoch_val_loss = 0.0
            num_val_batches = 0
            all_val_preds = []
            all_val_labels = []
            
            with torch.no_grad():
                for batch in val_dataloader_kd:
                    audio_features = batch['audio_features'].to(device)
                    hard_labels = batch['hard_label'].to(device)
                    soft_labels = batch['soft_label'].to(device)
                    
                    student_logits = student_model_iter(audio_features)
                    
                    loss_dict = compute_kd_loss_with_soft_targets(
                        student_logits=student_logits,
                        soft_targets=soft_labels,
                        hard_labels=hard_labels,
                        temperature=Config.TEMPERATURE,
                        alpha_ce=Config.ALPHA_CE,
                        alpha_kl=Config.ALPHA_KL
                    )
                    
                    epoch_val_loss += loss_dict['total_loss'].item()
                    num_val_batches += 1
                    
                    all_val_preds.extend(torch.argmax(student_logits, dim=1).cpu().numpy())
                    all_val_labels.extend(hard_labels.cpu().numpy())
            
            avg_val_loss = epoch_val_loss / num_val_batches
            val_accuracy = accuracy_score(all_val_labels, all_val_preds)
            
            scheduler_iter.step(avg_val_loss)
            
            if (epoch + 1) % 5 == 0:
                print(f"   Epoch {epoch+1}/{num_epochs_iter}: Val Loss={avg_val_loss:.4f}, Val Acc={val_accuracy*100:.2f}%")
            
            # Checkpoint
            if avg_val_loss < best_val_loss_iter:
                best_val_loss_iter = avg_val_loss
                patience_counter_iter = 0
                torch.save(student_model_iter.state_dict(), 
                          checkpoint_dir_iter / "best_model.pth")
            else:
                patience_counter_iter += 1
                if patience_counter_iter >= Config.EARLY_STOPPING_PATIENCE_STUDENT:
                    print(f"   Early stopping at epoch {epoch+1}")
                    break
        
        # Load best model
        student_model_iter.load_state_dict(
            torch.load(checkpoint_dir_iter / "best_model.pth", map_location=device)
        )
        
        # ---- VALUTA STUDENT SU TEST SET ----
        print(f"\n📊 Evaluating iteration {iteration} on test set")
        
        student_model_iter.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in test_dataloader_kd:
                audio_features = batch['audio_features'].to(device)
                hard_labels = batch['hard_label'].cpu().numpy()
                
                student_logits = student_model_iter(audio_features)
                predictions = torch.argmax(student_logits, dim=1).cpu().numpy()
                
                all_preds.extend(predictions)
                all_labels.extend(hard_labels)
        
        iter_accuracy = accuracy_score(all_labels, all_preds)
        iter_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        
        iteration_history['iteration'].append(iteration)
        iteration_history['test_accuracy'].append(iter_accuracy)
        iteration_history['test_f1'].append(iter_f1)
        
        print(f"\n   ✅ Iteration {iteration} Results:")
        print(f"      - Test Accuracy: {iter_accuracy*100:.2f}%")
        print(f"      - Test F1-Score: {iter_f1:.4f}")
        
        # Per prossima iterazione, usa questo student
        student_model = student_model_iter
    
    # ---- RIASSUNTO ITERAZIONI ----
    print(f"\n\n" + "="*80)
    print("📊 ITERATIVE TRAINING SUMMARY")
    print("="*80)
    
    print(f"\nIteration Results:")
    for i in range(len(iteration_history['iteration'])):
        print(f"   Iteration {iteration_history['iteration'][i]}: " +
              f"Accuracy={iteration_history['test_accuracy'][i]*100:.2f}%, " +
              f"F1={iteration_history['test_f1'][i]:.4f}")

print("\n" + "="*80)
print("✅ ITERATIVE NOISY STUDENT COMPLETED")
print("="*80 + "\n")
